In [ ]:
!pip install chromadb sentence-transformers langchain_community langchain_core langgraph langchain-google-genai langchain-text-splitters pypdf

In [ ]:
import os
import pandas as pd
import chromadb
import pypdf
from typing import TypedDict, List
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
import random

from langchain_core.tools import tool
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
past_loans_df=pd.read_csv("/content/past_loans.csv")
print("Initializing Embeddings Model...")
embeddings_model = HuggingFaceEmbeddings(model_name="mixedbread-ai/mxbai-embed-large-v1")

In [ ]:
def embed_text(text: str) -> list[float]:
    """Generate embeddings for the given text using LangChain."""
    return embeddings_model.embed_query(text)

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "langmsith_key"
os.environ["LANGCHAIN_PROJECT"] = "Loan_Eligibility_Multi_Agent"
os.environ['GOOGLE_API_KEY'] = 'api_key'
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', max_retries=3)

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_banking_store")
rules_collection = chroma_client.get_or_create_collection("lending_rules")
past_loans_collection = chroma_client.get_or_create_collection("past_loans")

In [ ]:
def initialize_knowledge_base(policy_pdf_path: str, past_loans_csv_path: str):
    """Reads existing files and populates ChromaDB if empty."""

    # --- A. Ingest Lending Policy Guidelines PDF ---
    if rules_collection.count() == 0:
        if os.path.exists(policy_pdf_path):
            print(f"📄 Processing Policy PDF: {policy_pdf_path}...")
            loader = PyPDFLoader(policy_pdf_path)
            docs = loader.load()

            # Split PDF into smaller chunks so the RAG tool can retrieve specific rules
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
            chunks = text_splitter.split_documents(docs)

            texts = [chunk.page_content for chunk in chunks]
            ids = [f"rule_chunk_{i}" for i in range(len(texts))]
            embeddings = embeddings_model.embed_documents(texts)

            rules_collection.add(documents=texts, ids=ids, embeddings=embeddings)
            print(f"Inserted {len(texts)} policy chunks into Vector DB.")
        else:
            print(f"File not found: {policy_pdf_path}")

    # --- B. Ingest Past Loans CSV ---
    if past_loans_collection.count() == 0:
        if os.path.exists(past_loans_csv_path):
            print(f"Processing Past Loans CSV: {past_loans_csv_path}...")
            df = pd.read_csv(past_loans_csv_path)

            texts = []
            ids = []

            for idx, row in df.iterrows():
                # Convert the raw row into a semantic string for better embeddings
                text = (f"Applicant is a {row['age']} year old {row['employment_type']} earning {row['monthly_income']} monthly. "
                        f"Credit score: {row['credit_score']}. Existing loans: {row.get('existing_loans', 'Unknown')}. "
                        f"Repayment history: {row.get('repayment_history', 'Unknown')}. "
                        f"Requested {row['loan_amount_requested']} for {row['loan_purpose']}. "
                        f"Historical Outcome: {row['risk_level']} risk, Status: {row['eligibility_status']}.")

                texts.append(text)
                ids.append(str(row['application_id']))

            embeddings = embeddings_model.embed_documents(texts)
            past_loans_collection.add(documents=texts, ids=ids, embeddings=embeddings)
            print(f"Inserted {len(texts)} historical applications into Vector DB.")
        else:
            print(f"File not found: {past_loans_csv_path}")

# Run the ingestion function
initialize_knowledge_base(
    policy_pdf_path="/content/Lending Policy Guidelines.pdf",
    past_loans_csv_path="/content/past_loans.csv"
)

In [ ]:
past_loan_data = past_loans_collection.get()
rules_data=rules_collection.get()

print(past_loan_data)
print(rules_data)

In [ ]:
class ApplicantInfo(BaseModel):
    customer_name: str = Field(description="Name of applicant")
    age: int = Field(description="Age")
    pan_num: str = Field(description="The PAN CARD number. If not explicitly written in the text, output 'UNKNOWN'.")
    monthly_income: float = Field(description="Monthly income in INR")
    employment_type: str = Field(description="Salaried or Self-Employed")
    credit_score: int = Field(description="Credit score")
    existing_loans: str = Field(description="Describe existing loans (e.g., 'Yes, 1 personal loan' or 'No')")
    loan_amount_requested: float = Field(description="Loan amount in INR")
    loan_purpose: str = Field(description="Purpose of loan")
    document_status: str = Field(description="'Complete' if ID, Salary Slips, and Bank Statements are present. Otherwise 'Pending'")
    missing_docs_summary: str = Field(description="If document_status is Pending, list what is missing. Else 'All essential documents verified.'")

class FraudAndVerificationMetrics(BaseModel):
    fraud_probability: float = Field(description="Simulated fraud probability between 0.0 and 1.0")
    fraud_flags: str = Field(description="Specific anomalies flagged (e.g., 'Income mismatch', 'None')")
    employment_verification_status: str = Field(description="Simulated result (e.g., 'Verified via HR', 'Pending verification')")

class FinancialMetrics(BaseModel):
    dti_ratio: float = Field(description="Calculated or estimated Debt-to-Income percentage")
    statistical_summary: str = Field(description="A 3-sentence statistical analysis of their financial health, DTI, and credit standing.")

@tool("search_rules_and_policies")
def search_rules_and_policies(query: str) -> str:
    """Search for risk assessment rules and lending policies."""
    results = rules_collection.query(query_embeddings=[embed_text(query)], n_results=3)
    return " \n".join([str(item) for sublist in results["documents"] for item in sublist])

@tool("find_similar_historical_applications")
def find_similar_historical_applications(summary: str) -> str:
    """Search for past loan applications similar to the current applicant."""
    results = past_loans_collection.query(query_embeddings=[embed_text(summary)], n_results=3)
    return " \n".join([str(item) for sublist in results["documents"] for item in sublist])

llm_with_tools = llm.bind_tools([search_rules_and_policies, find_similar_historical_applications])

In [ ]:
class ApplicationState(TypedDict):
    application_id: str
    pdf_file_path: str
    applicant_data: dict
    document_status: str
    missing_docs_summary: str
    financial_summary: str
    verification_summary: str
    fraud_probability: float
    risk_assessment: str
    recommendation: str
    applicant_decision: str
    final_status: str

In [ ]:
def pdf_ingestion_node(state: ApplicationState):
    """Extracts data and strictly verifies essential documents."""
    print(f"-> RUNNING: PDF Ingestion Node (Reading {state['pdf_file_path']})")
    try:
        loader = PyPDFLoader(state["pdf_file_path"])
        pdf_text = "\n".join([page.page_content for page in loader.load()])
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return {"document_status": "Pending", "missing_docs_summary": "PDF file could not be read.", "applicant_data": {}}

    prompt = f"""
    PERSONA: You are a strict Document Verification AI.
    INSTRUCTION: Read the following applicant text. Check if ALL essential documents are mentioned as attached/verified (ID Proof, Salary Slips, Bank Statements).
    If ANY are missing, set document_status to 'Pending' and summarize what is missing in missing_docs_summary.
    If all are present, set document_status to 'Complete'.

    APPLICANT TEXT:
    {pdf_text}
    """

    structured_llm = llm.with_structured_output(ApplicantInfo)
    extracted_data = structured_llm.invoke([HumanMessage(content=prompt)]).model_dump()

    return {
        "applicant_data": extracted_data,
        "document_status": extracted_data.get("document_status", "Pending"),
        "missing_docs_summary": extracted_data.get("missing_docs_summary", "Verification failed.")
    }

# initial_state = {
#         "application_id": "NEW_APP_001",
#         "pdf_file_path": "/content/mock_new_applicant.pdf"
#     }
# res=pdf_ingestion_node(initial_state)
# res

In [ ]:
def fetch_credit_bureau_data(pan_num: str) -> dict:
    """Mock API to simulate pulling a CIBIL/Experian credit report."""
    print(f"📡 [API CALL] Fetching Credit Bureau Data for ID: {pan_num}...")

    if pan_num == "UNKNOWN":
        return {
            "credit_score": 0,
            "existing_loans": "Unknown",
            "repayment_history": "API FAILURE: Cannot pull report without PAN CARD NUMBER."
        }

    database = {
        "PAN-BAD-001": {"credit_score": 450, "existing_loans": "3 active personal loans", "repayment_history": "Poor - Defaults."},
        "PAN-GOOD-002": {"credit_score": 780, "existing_loans": "None", "repayment_history": "Excellent"}
    }
    return database.get(pan_num, {"credit_score": 650, "existing_loans": "None", "repayment_history": "Average"})

In [ ]:
def financial_analysis_node(state: ApplicationState):
    """Calculates DTI and pulls background credit data."""
    print("-> RUNNING PARALLEL: Financial Analysis Node (Calculating Metrics...)")
    data = state.get("applicant_data", {})

    # 1. PULL ACTUAL CREDIT DATA VIA MOCK API
    pan_num = data.get("pan_num", "UNKNOWN")
    credit_report = fetch_credit_bureau_data(pan_num)

    prompt = f"""
    PERSONA: You are a Quantitative Financial Analyst AI.
    CONTEXT: You need to perform statistical analysis for a loan applicant.

    APPLICANT DECLARED DATA: {data}
    CREDIT BUREAU REPORT: {credit_report}

    INSTRUCTIONS:
    1. Estimate their existing monthly debt obligations based on the 'existing_loans' from the BUREAU REPORT.
    2. Calculate the Debt-to-Income (DTI) ratio percentage: (Estimated Debt / Monthly Income) * 100.
    3. Write a 3-sentence statistical summary combining what they declared vs their actual credit report.

    OUTPUT: Provide ONLY the requested structured JSON metrics.
    """

    structured_llm = llm.with_structured_output(FinancialMetrics)
    metrics = structured_llm.invoke([HumanMessage(content=prompt)]).model_dump()

    # Combine everything into a master summary for the Risk Agent
    summary = (
        f"{metrics['statistical_summary']}\n"
        f"Financial Metrics:\n"
        f"- DTI Ratio: {metrics['dti_ratio']}%\n"
        f"- Actual Credit Score (Bureau): {credit_report['credit_score']}\n"
        f"- Repayment History (Bureau): {credit_report['repayment_history']}\n"
    )

    return {"financial_summary": summary}

In [ ]:
def fraud_and_verification_node(state: ApplicationState):
    """Simulates Fraud checks and Employment verification using advanced prompting."""
    print("-> RUNNING PARALLEL: Fraud & Employment Verification Node")
    data = state.get("applicant_data", {})

    prompt = f"""
    PERSONA/ROLE: You are an Expert Fraud Investigator and KYC Specialist at a Tier-1 Bank.

    CONTEXT:
    You are performing a parallel background check on a new loan applicant while the finance team calculates their metrics.
    You do not have access to live databases, so you must *simulate* the investigation based on standard risk patterns.

    APPLICANT DATA:
    {data}

    INSTRUCTIONS:
    1. Simulate an Employment Verification check. (e.g., If Salaried, simulate calling the employer. If Self-Employed, simulate checking tax registrations).
    2. Analyze the profile for anomalies to generate a Fraud Probability Score (0.0 to 1.0).
    3. Identify any 'Fraud Flags'.

    EXAMPLES OF ANOMALIES:
    - High requested loan amount with very low income and poor credit score -> High Fraud Risk (0.7 - 0.9).
    - EXPLICIT FRAUD RULE: If the applicant's data says ID Proof is attached (document_status is Complete) BUT the 'pan_num' is 'UNKNOWN', this is Identity Obfuscation. You MUST assign a Fraud Probability of 0.85 or higher!
    - Unusually young age (e.g., 19) requesting a massive business loan -> Medium/High Fraud Risk.
    - Salaried with good credit and proportionate loan -> Low Fraud Risk (0.0 - 0.1).

    GUARDRAILS:
    - fraud_probability MUST be a float between 0.0 and 1.0.
    - Do NOT make up external personal details (like fake company names); keep the simulation generic (e.g., "HR department confirmed").

    TONE: Objective, cautious, strictly analytical.
    """

    # Enforce structured JSON output
    structured_llm = llm.with_structured_output(FraudAndVerificationMetrics)
    metrics = structured_llm.invoke([HumanMessage(content=prompt)]).model_dump()

    # Format the output into a readable string for the Risk Agent later
    verification_summary = (
        f"--- FRAUD & KYC VERIFICATION ---\n"
        f"Fraud Probability: {metrics['fraud_probability']}\n"
        f"Flags Detected: {metrics['fraud_flags']}\n"
        f"Employment Status: {metrics['employment_verification_status']}\n"
    )

    # Notice we return fraud_probability natively so the router or risk agent can use the math!
    return {
        "verification_summary": verification_summary,
        "fraud_probability": metrics["fraud_probability"]
    }

In [ ]:
def stage_1_sync(state: ApplicationState):
    """Merges the parallel node outputs before routing."""
    return {}

In [ ]:
def risk_assessment_node(state: ApplicationState):
    """Performs heavy risk analysis utilizing prompt engineering, RAG, and financial stats."""
    print("-> RUNNING: Risk Assessment Node (Consulting RAG Policies...)")

    fin_summary = state.get("financial_summary", "No financial summary provided.")
    verifications = state.get("verification_summary", "No verification data provided.")
    data = state.get("applicant_data", {})

    sys_prompt = """
    PERSONA: You are a Senior Chief Risk Officer at a Tier-1 Bank.
    ROLE: Perform absolute, objective risk analysis on loan applications.

    CONTEXT:
    You have access to the applicant's raw data, financial analysis, and simulated verifications.
    You MUST use your provided tools to search the vector database for current 'Lending Policies' and 'Historical Precedents'.

    INSTRUCTIONS:
    1. Use 'search_rules_and_policies' to find rules matching the applicant's credit score and requested amount.
    2. Use 'find_similar_historical_applications' to see how similar profiles were handled.
    3. Cross-reference the retrieved rules with the applicant's DTI, Credit Score, and Fraud Risk.

    GUARDRAILS:
    - NEVER approve or reject the loan yourself. Only output a Risk Evaluation (LOW, MEDIUM, or HIGH).
    - If Simulated Fraud Risk > 0.30, you MUST categorize as HIGH RISK.
    - If the applicant violates a retrieved lending policy, clearly state which one.

    OUTPUT FORMAT:
    Produce a 4-paragraph report:
    1. Precedent Analysis (What historical data showed)
    2. Policy Compliance (Which rules were passed/failed)
    3. Quantitative Risk (Analysis of DTI and Credit)
    4. Final Risk Tier (LOW/MEDIUM/HIGH) with a 1-sentence justification.

    TONE: Highly professional, objective, analytical, and uncompromising.
    """

    user_prompt = f"Evaluate this applicant:\nRaw Data: {data}\nFinancial Analysis: {fin_summary}\nVerification Data: {verifications}"

    messages = [SystemMessage(content=sys_prompt), HumanMessage(content=user_prompt)]
    response = llm_with_tools.invoke(messages)

    # Tool Execution Loop
    if response.tool_calls:
        messages.append(response)
        for tool_call in response.tool_calls:
            if tool_call["name"] == "search_rules_and_policies":
                res = search_rules_and_policies.invoke(tool_call["args"])
            else:
                res = find_similar_historical_applications.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(res), tool_call_id=tool_call["id"]))

        final_response = llm.invoke(messages)
        risk_text = final_response.content
    else:
        risk_text = response.content

    return {"risk_assessment": risk_text}

In [ ]:
def recommendation_node(state: ApplicationState):
    """Generates the Loan Offer."""
    print("-> RUNNING: Recommendation Node")
    if state.get("document_status") == "Pending":
        reason = state.get("missing_docs_summary", "Essential documents missing.")
        return {"recommendation": "STATUS HOLD: Documents are pending or missing. 0 amount approved until documents are submitted."}

    prompt = f"Generate a brief, professional loan offer for {state['applicant_data'].get('customer_name')}. They requested {state['applicant_data'].get('loan_amount_requested')}. Based on this risk assessment: {state.get('risk_assessment')}. State the Approved Amount and Proposed Interest Rate %."
    return {"recommendation": llm.invoke([HumanMessage(content=prompt)]).content}

def human_agent_escalation_node(state: ApplicationState):
    """Triggered if applicant rejects terms."""
    print("-> RUNNING: Escalation Node")
    ticket_number = f"TKT-{random.randint(100000, 999999)}"
    return {
        "final_status": (
            f"ESCALATED TO HUMAN REP.\n"
            f"Ticket Number: {ticket_number}\n"
            f"Dear Customer, a ticket has been created for manual renegotiation. "
            f"Our representatives will contact you shortly."
        )
    }


In [ ]:
def router_after_sync(state: ApplicationState):
    """Router: Check for missing docs."""
    if state.get("document_status") == "Pending":
        print("[Router] Documents Pending. Auto-Routing to Hold (Skipping Risk).")
        return "recommendation_node"
    print("[Router] Documents Complete. Routing to Risk Assessment.")
    return "risk_assessment_node"

def router_after_review(state: ApplicationState):
    """HITL Router: Did applicant accept or escalate?"""
    if state["applicant_decision"] == "escalate":
        return "human_agent_escalation_node"
    return END

workflow = StateGraph(ApplicationState)

# Add Nodes
workflow.add_node("pdf_ingestion_node", pdf_ingestion_node)
workflow.add_node("financial_analysis_node", financial_analysis_node)
workflow.add_node("fraud_and_verification_node", fraud_and_verification_node)
workflow.add_node("stage_1_sync", stage_1_sync)
workflow.add_node("risk_assessment_node", risk_assessment_node)
workflow.add_node("recommendation_node", recommendation_node)
workflow.add_node("human_agent_escalation_node", human_agent_escalation_node)


workflow.set_entry_point("pdf_ingestion_node")

# 1. PARALLEL FAN-OUT
workflow.add_edge("pdf_ingestion_node", "financial_analysis_node")
workflow.add_edge("pdf_ingestion_node", "fraud_and_verification_node")

# 2. PARALLEL FAN-IN (SYNC)
workflow.add_edge("financial_analysis_node", "stage_1_sync")
workflow.add_edge("fraud_and_verification_node", "stage_1_sync")

# 3. ROUTER (WITH EXPLICIT PATH MAPPING FOR THE VISUALIZER)
workflow.add_conditional_edges(
    "stage_1_sync",
    router_after_sync,
    {
        "recommendation_node": "recommendation_node",
        "risk_assessment_node": "risk_assessment_node"
    }
)

# 4. SEQUENTIAL
workflow.add_edge("risk_assessment_node", "recommendation_node")

# 5. HUMAN-IN-THE-LOOP ROUTER (WITH EXPLICIT PATH MAPPING)
workflow.add_conditional_edges(
    "recommendation_node",
    router_after_review,
    {
        "human_agent_escalation_node": "human_agent_escalation_node",
        END: END
    }
)

workflow.add_edge("human_agent_escalation_node", END)

# Compile with Memory
memory = MemorySaver()
app_graph = workflow.compile(checkpointer=memory, interrupt_after=["recommendation_node"])

In [ ]:
app_graph

In [ ]:
import os

def run_workflow():
    # 1. API Key Check
    if not os.environ.get("GOOGLE_API_KEY"):
        print("\n❌ FATAL ERROR: Please set your GOOGLE_API_KEY environment variable!")
        return

    # 2. Setup Thread Config (CRITICAL for Human-in-the-Loop memory)
    thread_config = {"configurable": {"thread_id": "DEMO_APP_001"}}

    # 3. Define the Initial State
    initial_state = {
        "application_id": "DEMO_APP_001",
        "pdf_file_path": "/content/test_2_bad_credit.pdf",
        "applicant_data": {},
        "document_status": "",
        "missing_docs_summary": "",
        "financial_summary": "",
        "fraud_probability": 0.0,
        "verification_summary": "",
        "risk_assessment": "",
        "recommendation": "",
        "applicant_decision": "",
        "final_status": ""
    }

    print("AI LOAN ELIGIBILITY SYSTEM INITIALIZED")
    print("==============================================\n")

    # 4. PHASE 1: Run Graph until the Recommendation Node (Pause Point)
    print("PROCESSING APPLICATION...")
    # stream() yields the output of each node as it runs
    for event in app_graph.stream(initial_state, config=thread_config):
        for node_name, node_output in event.items():
            print(f"✅ Completed: {node_name}")

    # 5. PHASE 2: HUMAN-IN-THE-LOOP (HITL) CHECKPOINT
    # Fetch the state exactly where the graph paused
    current_state = app_graph.get_state(thread_config).values

    print("\n----------------------------------------------")
    print(" AI LOAN OFFER & RISK SUMMARY:")
    print("----------------------------------------------")
    print(current_state.get("recommendation", "No recommendation generated."))
    print("----------------------------------------------\n")

    # Ask the human for input with retry logic
    max_attempts = 3
    attempt = 0

    while attempt < max_attempts:
        user_input = input(
            "\nAPPLICANT ACTION:\n"
            "Type 'yes' or 'accept' to proceed.\n"
            "Type 'no' or 'escalate' to escalate to a human representative.\n> "
        ).strip().lower()

        # ACCEPT CASES
        if user_input in ['yes', 'accept']:
            user_input = 'accept'
            break

        # ESCALATE CASES
        elif user_input in ['no', 'escalate']:
            user_input = 'escalate'
            break

        # INVALID INPUT
        else:
            attempt += 1

            if attempt < max_attempts:
                print("Invalid input. Please try again.")
            else:
                print("Maximum attempts reached. Defaulting to 'escalate'.")
                user_input = 'escalate'

    # Update the graph's state memory with the human's decision
    app_graph.update_state(thread_config, {"applicant_decision": user_input})

    # 6. PHASE 3: RESUME AND FINISH GRAPH
    print(f"\n-> Resuming workflow with decision: '{user_input.upper()}'...")

    # Passing 'None' as the input tells LangGraph to resume from its last checkpoint!
    for event in app_graph.stream(None, config=thread_config):
        for node_name, node_output in event.items():
            print(f"✅ Completed: {node_name}")

    # 7. FINAL OUTCOME
    final_state = app_graph.get_state(thread_config).values

    print("\n==============================================")
    print("WORKFLOW FINISHED")
    print("==============================================")

    if user_input == 'escalate':
        print(f"FINAL OUTCOME: {final_state.get('final_status', 'Escalation recorded.')}")
    else:
        print("FINAL OUTCOME: LOAN APPROVED & ACCEPTED BY USER. Contract generated.")

# Execute the test
if __name__ == "__main__":
    run_workflow()

In [ ]:
!pip install fpdf

In [ ]:
from fpdf import FPDF

def create_mock_pdf(filename, text):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    for line in text.split('\n'):
        # Handle cell text properly
        pdf.multi_cell(0, 10, txt=line)
    pdf.output(filename)
    print(f"✅ Created Test File: {filename}")

# =======================================================
# 🟢 CASE 1: THE PERFECT APPLICANT (Happy Path -> Approval)
# =======================================================
case1_text = """LOAN APPLICATION FORM
----------------------------------
Applicant Name: Anjali Desai
Age: 32
PAN Card Number: PAN-GOOD-002
Employment: Salaried
Monthly Income: 120000
LOAN DETAILS
----------------------------------
Requested Amount: 1500000
Purpose: Home Loan
DECLARATION & DOCUMENTS
----------------------------------
ID Proof: Attached
Salary Slips: Attached
Bank Statements: Attached
Status: All documents successfully verified and attached."""
create_mock_pdf("test_1_perfect_applicant.pdf", case1_text)

# =======================================================
# 🔴 CASE 2: BAD CREDIT (Fails at Risk Agent via API/RAG)
# =======================================================
case2_text = """LOAN APPLICATION FORM
----------------------------------
Applicant Name: John Doe
Age: 45
PAN Card Number: PAN-BAD-001
Employment: Salaried
Monthly Income: 150000
LOAN DETAILS
----------------------------------
Requested Amount: 2000000
Purpose: Personal Loan
DECLARATION & DOCUMENTS
----------------------------------
ID Proof: Attached
Salary Slips: Attached
Bank Statements: Attached
Status: All documents successfully verified and attached."""
create_mock_pdf("test_2_bad_credit.pdf", case2_text)

# =======================================================
# 🟡 CASE 3: MISSING DOCUMENTS (Fails at PDF Node -> Hold)
# =======================================================
case3_text = """LOAN APPLICATION FORM
----------------------------------
Applicant Name: Rahul Sharma
Age: 29
PAN Card Number: PAN-GOOD-002
Employment: Salaried
Monthly Income: 85000
LOAN DETAILS
----------------------------------
Requested Amount: 500000
Purpose: Vehicle Loan
DECLARATION & DOCUMENTS
----------------------------------
ID Proof: Attached
Salary Slips: Attached
Bank Statements: MISSING - Will submit next week
Status: Documents pending."""
create_mock_pdf("test_3_missing_docs.pdf", case3_text)

# =======================================================
# 🕵️‍♂️ CASE 4: IDENTITY OBFUSCATION (Fails at Fraud Node -> Auto Reject)
# =======================================================
case4_text = """LOAN APPLICATION FORM
----------------------------------
Applicant Name: Ramesh Kumar
Age: 35
PAN Card Number:
Employment: Self-Employed
Monthly Income: 90000
LOAN DETAILS
----------------------------------
Requested Amount: 1000000
Purpose: Business Loan
DECLARATION & DOCUMENTS
----------------------------------
ID Proof: Attached
Salary Slips: Attached
Bank Statements: Attached
Status: All documents successfully verified and attached."""
create_mock_pdf("test_4_identity_obfuscation.pdf", case4_text)

# =======================================================
# 🤡 CASE 5: BLATANT DATA FRAUD (Fails at Fraud Node -> Auto Reject)
# =======================================================
case5_text = """LOAN APPLICATION FORM
----------------------------------
Applicant Name: Tim David
Age: 18
PAN Card Number: PAN-GOOD-002
Employment: Unemployed
Monthly Income: 0
LOAN DETAILS
----------------------------------
Requested Amount: 50000000
Purpose: Vacation
DECLARATION & DOCUMENTS
----------------------------------
ID Proof: Attached
Salary Slips: Attached
Bank Statements: Attached
Status: All documents successfully verified and attached."""
create_mock_pdf("test_5_blatant_fraud.pdf", case5_text)